In [5]:
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np
from torch.utils.data import DataLoader, TensorDataset, random_split

# Load data
X = np.load("../data/AlbaniaSAT/processed/patches.npy")
y = np.load("../data/AlbaniaSAT/processed/labels.npy")

# RGB only, normalize
X_rgb = X[:, :3, :, :]
X_rgb = np.clip(X_rgb, 0, 3000) / 3000.0
mean = np.array([0.485, 0.456, 0.406]).reshape(1, 3, 1, 1)
std = np.array([0.229, 0.224, 0.225]).reshape(1, 3, 1, 1)
X_rgb = (X_rgb - mean) / std

X_tensor = torch.tensor(X_rgb, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

# Split 70/15/15
dataset = TensorDataset(X_tensor, y_tensor)
n = len(dataset)
n_train = int(0.7 * n)
n_val = int(0.15 * n)
n_test = n - n_train - n_val

train_set, val_set, test_set = random_split(
    dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False)

print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

Train: 2800 | Val: 600 | Test: 600


In [6]:
# Load EuroSAT pretrained weights
checkpoint = torch.load("../results/models/resnet50_eurosat.pth",
                        map_location="cpu")
model = models.resnet50()
model.fc = nn.Linear(2048, 10)
model.load_state_dict(checkpoint)

# Freeze everything
for param in model.parameters():
    param.requires_grad = False

# Replace classifier with 8-class head
model.fc = nn.Linear(2048, 8)

print("Model loaded and ready for Stage 1!")

Model loaded and ready for Stage 1!


In [7]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [8]:
def train_stage(model, train_loader, val_loader, epochs, lr, stage_name):
    model = model.to(device)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr
    )
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    
    best_val_acc = 0
    best_weights = None
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_correct = 0
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            train_correct += (outputs.argmax(1) == y_batch).sum().item()
        
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                val_correct += (outputs.argmax(1) == y_batch).sum().item()
        
        train_acc = train_correct / len(train_loader.dataset) * 100
        val_acc = val_correct / len(val_loader.dataset) * 100
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        
        scheduler.step()
        print(f"{stage_name} | Epoch {epoch+1}/{epochs} | "
              f"Train: {train_acc:.2f}% | Val: {val_acc:.2f}%")
    
    model.load_state_dict(best_weights)
    model = model.to(device)
    print(f"\n{stage_name} done! Best val accuracy: {best_val_acc:.2f}%\n")
    return model

In [10]:
# Stage 1 — classifier head only
print("=" * 50)
print("STAGE 1 — Training classifier head only")
print("=" * 50)
model = train_stage(model, train_loader, val_loader, 
                    epochs=5, lr=1e-3, stage_name="Stage 1")

# Stage 2 — unfreeze layer4
print("=" * 50)
print("STAGE 2 — Unfreezing layer4")
print("=" * 50)
for param in model.layer4.parameters():
    param.requires_grad = True
model = train_stage(model, train_loader, val_loader,
                    epochs=5, lr=1e-4, stage_name="Stage 2")

# Stage 3 — unfreeze layer3
print("=" * 50)
print("STAGE 3 — Unfreezing layer3")
print("=" * 50)
for param in model.layer3.parameters():
    param.requires_grad = True
model = train_stage(model, train_loader, val_loader,
                    epochs=5, lr=1e-5, stage_name="Stage 3")

# Save final model
torch.save(model.state_dict(), "../results/models/resnet50_albaniasat.pth")
print("Model saved!")

STAGE 1 — Training classifier head only
Stage 1 | Epoch 1/5 | Train: 39.79% | Val: 51.00%
Stage 1 | Epoch 2/5 | Train: 48.18% | Val: 51.67%
Stage 1 | Epoch 3/5 | Train: 51.68% | Val: 53.83%
Stage 1 | Epoch 4/5 | Train: 53.64% | Val: 59.17%
Stage 1 | Epoch 5/5 | Train: 53.93% | Val: 59.83%

Stage 1 done! Best val accuracy: 59.83%

STAGE 2 — Unfreezing layer4
Stage 2 | Epoch 1/5 | Train: 55.32% | Val: 60.83%
Stage 2 | Epoch 2/5 | Train: 62.64% | Val: 61.33%
Stage 2 | Epoch 3/5 | Train: 70.18% | Val: 63.33%
Stage 2 | Epoch 4/5 | Train: 77.96% | Val: 62.67%
Stage 2 | Epoch 5/5 | Train: 85.29% | Val: 62.33%

Stage 2 done! Best val accuracy: 63.33%

STAGE 3 — Unfreezing layer3
Stage 3 | Epoch 1/5 | Train: 78.82% | Val: 63.33%
Stage 3 | Epoch 2/5 | Train: 81.46% | Val: 62.33%
Stage 3 | Epoch 3/5 | Train: 83.54% | Val: 62.33%
Stage 3 | Epoch 4/5 | Train: 85.68% | Val: 62.50%
Stage 3 | Epoch 5/5 | Train: 87.11% | Val: 63.17%

Stage 3 done! Best val accuracy: 63.33%

Model saved!


## v2 — With Data Augmentation
Previous run (no augmentation) achieved 63.33% val accuracy.
This run adds random flips and rotations to reduce overfitting.

In [9]:
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np
from torch.utils.data import DataLoader, Dataset

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [10]:
X = np.load("../data/AlbaniaSAT/processed/patches.npy")
y = np.load("../data/AlbaniaSAT/processed/labels.npy")

X_rgb = X[:, :3, :, :]
X_rgb = np.clip(X_rgb, 0, 3000) / 3000.0
mean = np.array([0.485, 0.456, 0.406]).reshape(1, 3, 1, 1)
std = np.array([0.229, 0.224, 0.225]).reshape(1, 3, 1, 1)
X_rgb = (X_rgb - mean) / std
X_rgb = X_rgb.astype(np.float32)

print(f"Data shape: {X_rgb.shape}")

Data shape: (4000, 3, 64, 64)


In [11]:
class AlbaniaSATDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = X
        self.y = y
        self.augment = augment

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx], dtype=torch.float32)
        y = torch.tensor(self.y[idx], dtype=torch.long)
        if self.augment:
            if torch.rand(1) > 0.5:
                x = torch.flip(x, dims=[2])
            if torch.rand(1) > 0.5:
                x = torch.flip(x, dims=[1])
            k = torch.randint(0, 4, (1,)).item()
            x = torch.rot90(x, k, dims=[1, 2])
        return x, y

n = len(X_rgb)
indices = np.random.RandomState(42).permutation(n)
n_train = int(0.7 * n)
n_val   = int(0.15 * n)

train_idx = indices[:n_train]
val_idx   = indices[n_train:n_train + n_val]
test_idx  = indices[n_train + n_val:]

train_set = AlbaniaSATDataset(X_rgb[train_idx], y[train_idx], augment=True)
val_set   = AlbaniaSATDataset(X_rgb[val_idx],   y[val_idx],   augment=False)
test_set  = AlbaniaSATDataset(X_rgb[test_idx],  y[test_idx],  augment=False)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=32, shuffle=False)

print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")
print("Augmentation enabled on train set only!")

Train: 2800 | Val: 600 | Test: 600
Augmentation enabled on train set only!


In [12]:
def train_stage(model, train_loader, val_loader, epochs, lr, stage_name):
    model = model.to(device)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    best_val_acc = 0
    best_weights = None

    for epoch in range(epochs):
        model.train()
        train_correct = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_correct += (outputs.argmax(1) == y_batch).sum().item()

        model.eval()
        val_correct = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                val_correct += (outputs.argmax(1) == y_batch).sum().item()

        train_acc = train_correct / len(train_loader.dataset) * 100
        val_acc   = val_correct   / len(val_loader.dataset)   * 100

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        scheduler.step()
        print(f"{stage_name} | Epoch {epoch+1}/{epochs} | "
              f"Train: {train_acc:.2f}% | Val: {val_acc:.2f}%")

    model.load_state_dict(best_weights)
    model = model.to(device)
    print(f"\n{stage_name} done! Best val accuracy: {best_val_acc:.2f}%\n")
    return model

In [17]:
checkpoint = torch.load("../results/models/resnet50_eurosat.pth", map_location="cpu")
model = models.resnet50()
model.fc = nn.Linear(2048, 10)
model.load_state_dict(checkpoint)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(2048, 8)
model = model.to(device)

print("=" * 50)
print("STAGE 1 — Training classifier head only")
print("=" * 50)
model = train_stage(model, train_loader, val_loader,
                    epochs=5, lr=1e-3, stage_name="Stage 1")

print("=" * 50)
print("STAGE 2 — Unfreezing layer4")
print("=" * 50)
for param in model.layer4.parameters():
    param.requires_grad = True
model = train_stage(model, train_loader, val_loader,
                    epochs=5, lr=1e-4, stage_name="Stage 2")

print("=" * 50)
print("STAGE 3 — Unfreezing layer3")
print("=" * 50)
for param in model.layer3.parameters():
    param.requires_grad = True
model = train_stage(model, train_loader, val_loader,
                    epochs=5, lr=1e-5, stage_name="Stage 3")

torch.save(model.state_dict(), "../results/models/resnet50_albaniasat_v2.pth")
print("Model saved!")

STAGE 1 — Training classifier head only
Stage 1 | Epoch 1/5 | Train: 39.32% | Val: 50.50%
Stage 1 | Epoch 2/5 | Train: 48.93% | Val: 52.67%
Stage 1 | Epoch 3/5 | Train: 51.36% | Val: 50.50%
Stage 1 | Epoch 4/5 | Train: 52.43% | Val: 53.33%
Stage 1 | Epoch 5/5 | Train: 54.89% | Val: 53.17%

Stage 1 done! Best val accuracy: 53.33%

STAGE 2 — Unfreezing layer4
Stage 2 | Epoch 1/5 | Train: 55.82% | Val: 57.50%
Stage 2 | Epoch 2/5 | Train: 59.14% | Val: 59.50%
Stage 2 | Epoch 3/5 | Train: 61.64% | Val: 60.33%
Stage 2 | Epoch 4/5 | Train: 63.32% | Val: 62.33%
Stage 2 | Epoch 5/5 | Train: 64.54% | Val: 61.67%

Stage 2 done! Best val accuracy: 62.33%

STAGE 3 — Unfreezing layer3
Stage 3 | Epoch 1/5 | Train: 65.25% | Val: 61.83%
Stage 3 | Epoch 2/5 | Train: 65.57% | Val: 62.33%
Stage 3 | Epoch 3/5 | Train: 65.71% | Val: 62.50%
Stage 3 | Epoch 4/5 | Train: 66.14% | Val: 62.33%
Stage 3 | Epoch 5/5 | Train: 66.11% | Val: 63.50%

Stage 3 done! Best val accuracy: 63.50%

Model saved!


## v3 - SoftCon

In [16]:
from huggingface_hub import hf_hub_download

weights_path = hf_hub_download(
    repo_id="wangyi111/softcon",
    filename="B13_rn50_softcon.pth"
)

print(f"Weights ready: {weights_path}")

Weights ready: /Users/user/.cache/huggingface/hub/models--wangyi111--softcon/snapshots/bae909781911f8ec034b4b959992fae17b973c0c/B13_rn50_softcon.pth


In [17]:
# Load SoftCon with 13-band first layer, then adapt to 4 bands
model_softcon = models.resnet50()

state_dict = torch.load(weights_path, map_location="cpu")
if "model" in state_dict:
    state_dict = state_dict["model"]
elif "state_dict" in state_dict:
    state_dict = state_dict["state_dict"]

# Adapt conv1 from 13 bands to 4 bands
# Take the first 4 channel weights from the 13-band conv1
conv1_weight_13 = state_dict["conv1.weight"]  # shape [64, 13, 7, 7]
conv1_weight_4 = conv1_weight_13[:, :4, :, :]  # take first 4 bands

# Replace in state dict
state_dict["conv1.weight"] = conv1_weight_4

# Update model to accept 4 input channels
model_softcon.conv1 = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3, bias=False)

# Replace classifier
model_softcon.fc = nn.Identity()

# Load weights
missing, unexpected = model_softcon.load_state_dict(state_dict, strict=False)
print(f"Missing keys: {len(missing)}")
print(f"Unexpected keys: {len(unexpected)}")
print("SoftCon loaded with 4-band input!")

Missing keys: 0
Unexpected keys: 0
SoftCon loaded with 4-band input!


In [18]:
# Check what's in the dataloader
X_batch, y_batch = next(iter(train_loader))
print("Batch shape:", X_batch.shape)

Batch shape: torch.Size([32, 3, 64, 64])


In [19]:
# Rebuild dataset with 4 bands for SoftCon
X = np.load("../data/AlbaniaSAT/processed/patches.npy")
y = np.load("../data/AlbaniaSAT/processed/labels.npy")

# Use all 4 bands this time
X_4band = X[:, :4, :, :]  # B4, B3, B2, B8

# Normalize each band separately using SSL4EO-S12 statistics
# B4 (Red), B3 (Green), B2 (Blue), B8 (NIR)
mean_4 = np.array([0.485, 0.456, 0.406, 0.441]).reshape(1, 4, 1, 1)
std_4  = np.array([0.229, 0.224, 0.225, 0.220]).reshape(1, 4, 1, 1)

X_4band = np.clip(X_4band, 0, 3000) / 3000.0
X_4band = (X_4band - mean_4) / std_4
X_4band = X_4band.astype(np.float32)

# Rebuild splits
n = len(X_4band)
indices = np.random.RandomState(42).permutation(n)
n_train = int(0.7 * n)
n_val   = int(0.15 * n)

train_idx = indices[:n_train]
val_idx   = indices[n_train:n_train + n_val]
test_idx  = indices[n_train + n_val:]

train_set_4 = AlbaniaSATDataset(X_4band[train_idx], y[train_idx], augment=True)
val_set_4   = AlbaniaSATDataset(X_4band[val_idx],   y[val_idx],   augment=False)
test_set_4  = AlbaniaSATDataset(X_4band[test_idx],  y[test_idx],  augment=False)

train_loader_4 = DataLoader(train_set_4, batch_size=32, shuffle=True)
val_loader_4   = DataLoader(val_set_4,   batch_size=32, shuffle=False)
test_loader_4  = DataLoader(test_set_4,  batch_size=32, shuffle=False)

# Verify
X_batch, y_batch = next(iter(train_loader_4))
print("Batch shape:", X_batch.shape)
print("4-band dataloader ready!")

Batch shape: torch.Size([32, 4, 64, 64])
4-band dataloader ready!


In [20]:
# Reload SoftCon fresh
model_softcon = models.resnet50()

state_dict = torch.load(weights_path, map_location="cpu")
if "model" in state_dict:
    state_dict = state_dict["model"]
elif "state_dict" in state_dict:
    state_dict = state_dict["state_dict"]

# Adapt conv1 from 13 bands to 4 bands
conv1_weight_13 = state_dict["conv1.weight"]
conv1_weight_4 = conv1_weight_13[:, :4, :, :]
state_dict["conv1.weight"] = conv1_weight_4

# Update model to accept 4 input channels
model_softcon.conv1 = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3, bias=False)
model_softcon.fc = nn.Identity()

missing, unexpected = model_softcon.load_state_dict(state_dict, strict=False)

# Freeze everything
for param in model_softcon.parameters():
    param.requires_grad = False

# Replace classifier with 8-class head
model_softcon.fc = nn.Linear(2048, 8)
model_softcon = model_softcon.to(device)

print("=" * 50)
print("STAGE 1 — Training classifier head only")
print("=" * 50)
model_softcon = train_stage(model_softcon, train_loader_4, val_loader_4,
                    epochs=5, lr=1e-3, stage_name="Stage 1")

print("=" * 50)
print("STAGE 2 — Unfreezing layer4")
print("=" * 50)
for param in model_softcon.layer4.parameters():
    param.requires_grad = True
model_softcon = train_stage(model_softcon, train_loader_4, val_loader_4,
                    epochs=5, lr=1e-4, stage_name="Stage 2")

print("=" * 50)
print("STAGE 3 — Unfreezing layer3")
print("=" * 50)
for param in model_softcon.layer3.parameters():
    param.requires_grad = True
model_softcon = train_stage(model_softcon, train_loader_4, val_loader_4,
                    epochs=5, lr=1e-5, stage_name="Stage 3")

torch.save(model_softcon.state_dict(), "../results/models/resnet50_softcon_albaniasat.pth")
print("SoftCon model saved!")

STAGE 1 — Training classifier head only
Stage 1 | Epoch 1/5 | Train: 47.89% | Val: 57.83%
Stage 1 | Epoch 2/5 | Train: 56.86% | Val: 59.67%
Stage 1 | Epoch 3/5 | Train: 57.75% | Val: 60.17%
Stage 1 | Epoch 4/5 | Train: 58.68% | Val: 61.17%
Stage 1 | Epoch 5/5 | Train: 59.46% | Val: 60.33%

Stage 1 done! Best val accuracy: 61.17%

STAGE 2 — Unfreezing layer4
Stage 2 | Epoch 1/5 | Train: 58.68% | Val: 63.50%
Stage 2 | Epoch 2/5 | Train: 62.89% | Val: 63.00%
Stage 2 | Epoch 3/5 | Train: 65.93% | Val: 63.00%
Stage 2 | Epoch 4/5 | Train: 67.96% | Val: 63.17%
Stage 2 | Epoch 5/5 | Train: 70.32% | Val: 63.17%

Stage 2 done! Best val accuracy: 63.50%

STAGE 3 — Unfreezing layer3
Stage 3 | Epoch 1/5 | Train: 63.21% | Val: 63.50%
Stage 3 | Epoch 2/5 | Train: 64.00% | Val: 63.67%
Stage 3 | Epoch 3/5 | Train: 64.93% | Val: 64.83%
Stage 3 | Epoch 4/5 | Train: 66.61% | Val: 64.83%
Stage 3 | Epoch 5/5 | Train: 65.86% | Val: 63.83%

Stage 3 done! Best val accuracy: 64.83%

SoftCon model saved!
